# 02 · 研究面板 sanity check

目的：`build_panel` 在全量真实数据上生成了 `panel.parquet`。单元测试只能证明函数在手工例子上正确，
这里用**常识和真实历史事件**检查全量结果是否合理。

运行 `uv run python -m ashare.clean` 的输出：

| 项目 | 结果 | 判断 |
|---|---|---|
| 行数 | 12,490,773 | ✅ 5000+ 只股票 × 约 12 年 |
| 股票数 | 5,485（含退市） | ✅ |
| 日期范围 | 2014-01-02 ~ 2026-11-16 | ✅ |
| `ret` 为空 | 3 行 | ⚠️ 极少，下面单独检查 |

In [ ]:
from datetime import date
import polars as pl

panel = pl.read_parquet("../data/processed/panel.parquet")
panel.shape

## 1. 年度汇总

每年：平均每天股票池（`in_universe`）数量 / 单日最多涨停数 / 单日最多跌停数

In [ ]:
daily = panel.group_by("date").agg(
    n_universe=pl.col("in_universe").sum(),
    n_limit_up=pl.col("limit_up").sum(),
    n_limit_down=pl.col("limit_down").sum(),
)
yearly = (
    daily.group_by(pl.col("date").dt.year().alias("year"))
    .agg(
        avg_universe=pl.col("n_universe").mean().round(0),
        max_limit_up=pl.col("n_limit_up").max(),
        max_limit_down=pl.col("n_limit_down").max(),
    )
    .sort("year")
)
with pl.Config(tbl_rows=20):
    display(yearly)

### 观察

**涨跌停极值能对上真实市场事件**（说明涨跌停判断在真实数据上有效）：

| 年份 | 最多涨停 / 跌停 | 对应事件 |
|---|---|---|
| 2015 | 1554 / 2187 | 上半年牛市，6 月起股灾 |
| 2016 | 215 / 1437 | 年初熔断 |
| 2020 | 259 / 3214 | 春节后疫情首日开盘 |
| 2024 | 865 / 1454 | 年初小盘股暴跌；9 月底政策刺激大涨 |
| 2025 | 173 / 3037 | 4 月初关税冲击 |
| 2023 | 116 / 80 | 全年波动小，无极端交易日 |

**股票池数量**整体逐年增加（新股上市），2023 年后增速放缓（新股发行收紧）。

但 **2015 年（2193）少于 2014 年（2252）**：股灾期间大量公司主动申请停牌躲避下跌，
股票池要求"未停牌"，所以数量下降。这说明 `is_suspended` 在真实数据上起了作用；
如果不剔除停牌股，这段时期的回测会假设可以买卖实际上无法交易的股票。

## 2. 抽查：2019-02-25（当年涨停最多的一天）

用两种独立口径交叉验证涨停判断：我们的 `limit_up` vs 涨幅 ≥ 9.9%。

In [ ]:
d = panel.filter(pl.col("date") == date(2019, 2, 25))
d.select(
    n_limit_up=pl.col("limit_up").sum(),
    n_pct_ge_99=(pl.col("pct_chg") >= 9.9).sum(),
)

In [ ]:
# 多出来的几只：涨停但涨幅 < 9.9% 的股票
d.filter(pl.col("limit_up") & (pl.col("pct_chg") < 9.9)).select(
    "code", "board", "is_st", "pct_chg", "limit_pct"
)

结果：`limit_up` 307 只，涨幅 ≥ 9.9% 302 只。多出的 5 只是 **ST 股**（涨跌幅限制 5%），
涨停时涨幅约 5%，不会出现在 ≥ 9.9% 那一组。两种口径一致，且 ST 规则在真实数据上生效。

## 3. `ret` 为空的 3 行

In [ ]:
panel.filter(pl.col("ret").is_null()).select(
    "date", "code", "open", "close", "preclose", "volume", "tradestatus"
)

结论：（运行上面的单元格后在这里写下原因）。3 行占比极小，不影响研究。

## 结论

全量面板通过 sanity check：样本规模、日期范围合理；涨跌停判断与真实市场事件、独立口径一致；
停牌过滤在 2015 年股灾期间的影响清晰可见。可以进入阶段 2（因子库）。